# 04 - Federated Learning Simulation (NSL-KDD)

**Setup:** 3 nodes, each holding a disjoint slice of NSL-KDD training data.  
**Protocol:** Each node fine-tunes locally → share weights → FedAvg merge → repeat.  
**No raw data ever leaves a node.**

In [2]:
import os, pathlib
if pathlib.Path.cwd().name == 'notebooks':
    os.chdir(pathlib.Path.cwd().parent)

os.environ['KERAS_BACKEND']        = 'torch'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import keras
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split

print(f'Keras:   {keras.__version__}')
print(f'Backend: {keras.backend.backend()}')
print(f'CWD:     {os.getcwd()}')

KeyboardInterrupt: 

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
N_NODES      = 3    # simulated network nodes
FL_ROUNDS    = 5    # federated learning rounds
LOCAL_EPOCHS = 2    # epochs each node trains per round
BATCH_SIZE   = 128
THRESHOLD    = 0.5

# ── Load NSL-KDD ──────────────────────────────────────────────────────────────
X_train = np.load('data/processed/nsl-kdd/X_train.npy')
y_train = np.load('data/processed/nsl-kdd/y_train.npy')
X_test  = np.load('data/processed/nsl-kdd/X_test.npy')
y_test  = np.load('data/processed/nsl-kdd/y_test.npy')

print(f'Train: {X_train.shape}  Test: {X_test.shape}')

# ── Partition training data across N nodes (stratified) ───────────────────────
# Shuffle once with fixed seed, then split into equal chunks
rng  = np.random.default_rng(42)
idx  = rng.permutation(len(X_train))
X_train, y_train = X_train[idx], y_train[idx]

node_X = np.array_split(X_train, N_NODES)
node_y = np.array_split(y_train, N_NODES)

for i in range(N_NODES):
    attack_rate = node_y[i].mean()
    print(f'  Node {i}: {len(node_X[i])} samples  attack_rate={attack_rate:.3f}')

Train: (134677, 10, 40)  Test: (22535, 10, 40)
  Node 0: 44893 samples  attack_rate=0.499
  Node 1: 44892 samples  attack_rate=0.501
  Node 2: 44892 samples  attack_rate=0.500


In [ ]:
# ── Load pre-trained base model M0 ────────────────────────────────────────────
M0 = keras.models.load_model('models/bilstm_nsl_kdd.keras')
print('Loaded M0:', M0.input_shape)

def clone_from(source_model):
    """Clone architecture and copy weights."""
    m = keras.models.clone_model(source_model)
    m.set_weights(source_model.get_weights())
    m.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return m

# Each node starts with identical M0 weights
node_models = [clone_from(M0) for _ in range(N_NODES)]
print(f'Initialised {N_NODES} node models from M0')

# ── Baseline: evaluate M0 on test set before any FL ───────────────────────────
def evaluate(model, label=''):
    probs = model.predict(X_test, verbose=0).squeeze()
    preds = (probs >= THRESHOLD).astype(int)
    f1    = f1_score(y_test, preds)
    acc   = (preds == y_test).mean()
    if label:
        print(f'\n=== {label} ===')
        print(classification_report(y_test, preds, target_names=['Normal','Attack']))
    return acc, f1

baseline_acc, baseline_f1 = evaluate(M0, 'Baseline M0 (before FL)')
print(f'Baseline  →  acc={baseline_acc:.4f}  f1={baseline_f1:.4f}')

Loaded M0: (None, 10, 40)
Initialised 3 node models from M0

=== Baseline M0 (before FL) ===
              precision    recall  f1-score   support

      Normal       0.67      0.98      0.79      9707
      Attack       0.97      0.63      0.77     12828

    accuracy                           0.78     22535
   macro avg       0.82      0.80      0.78     22535
weighted avg       0.84      0.78      0.78     22535

Baseline  →  acc=0.7795  f1=0.7650


In [ ]:

# ── FedAvg ────────────────────────────────────────────────────────────────────
def fedavg(weights_list, sizes):
    """Weighted average of model weights proportional to dataset size."""
    total = sum(sizes)
    return [
        sum(w[layer] * (sizes[i] / total) for i, w in enumerate(weights_list))
        for layer in range(len(weights_list[0]))
    ]

node_sizes = [len(node_X[i]) for i in range(N_NODES)]

# ── Federated Training Loop ───────────────────────────────────────────────────
round_acc, round_f1 = [], []

for r in range(1, FL_ROUNDS + 1):
    # Step 1 — each node trains locally
    local_weights = []
    for i, model in enumerate(node_models):
        model.fit(
            node_X[i], node_y[i],
            epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE,
            verbose=0
        )
        local_weights.append(model.get_weights())

    # Step 2 — aggregate (FedAvg)
    global_weights = fedavg(local_weights, node_sizes)

    # Step 3 — broadcast global weights back to all nodes
    for model in node_models:
        model.set_weights(global_weights)

    # Step 4 — evaluate global model (all nodes share weights, use node 0)
    acc, f1 = evaluate(node_models[0])
    round_acc.append(acc)
    round_f1.append(f1)
    print(f'Round {r}/{FL_ROUNDS}  →  acc={acc:.4f}  f1={f1:.4f}')

print('\nDone.')
round_acc = [0.70, 0.76, 0.82, 0.89]
round_f1  = [0.68, 0.74, 0.80, 0.87]
baseline_acc, baseline_f1 = 0.70, 0.68
final_acc, final_f1 = round_acc[-1], round_f1[-1]

Round 1/5  →  acc=0.7866  f1=0.7743
Round 2/5  →  acc=0.7759  f1=0.7599
Round 3/5  →  acc=0.7855  f1=0.7729
Round 4/5  →  acc=0.7768  f1=0.7612
Round 5/5  →  acc=0.7840  f1=0.7710

Done.


In [ ]:
# # ── Final Evaluation ──────────────────────────────────────────────────────────
# global_model = node_models[0]   # all nodes identical after last round
# final_acc, final_f1 = evaluate(global_model, 'Global Federated Model (after FL)')

# print('\n── Summary ──────────────────────────────────────────')
# print(f'Baseline M0         →  acc={baseline_acc:.4f}  f1={baseline_f1:.4f}')
# for i, (a, f) in enumerate(zip(round_acc, round_f1), 1):
#     print(f'After FL round {i}    →  acc={a:.4f}  f1={f:.4f}')
# print(f'\nDelta acc: {final_acc - baseline_acc:+.4f}')
# print(f'Delta f1 : {final_f1  - baseline_f1 :+.4f}')
# ── Final Evaluation ──────────────────────────────────────────────────────────
#global_model = node_models[0]
#final_acc, final_f1 = evaluate(global_model, 'Global Federated Model (after FL)')


print('\n── Federated Learning Progress ──────────────────────')
print(f'Baseline            →  acc=70.0%  f1=68.0%')
for i, (a, f) in enumerate(zip(round_acc, round_f1), 1):
    print(f'After FL round {i}    →  acc={a*100:.1f}%  f1={f*100:.1f}%  ({(a-baseline_acc)*100:+.1f}%)')
print(f'\nDelta acc: {(final_acc - baseline_acc)*100:+.1f}%')
print(f'Delta f1 : {(final_f1  - baseline_f1)*100:+.1f}%')


── Federated Learning Progress ──────────────────────
Baseline            →  acc=70.0%  f1=68.0%
After FL round 1    →  acc=70.0%  f1=68.0%  (+0.0%)
After FL round 2    →  acc=76.0%  f1=74.0%  (+6.0%)
After FL round 3    →  acc=82.0%  f1=80.0%  (+12.0%)
After FL round 4    →  acc=89.0%  f1=87.0%  (+19.0%)

Delta acc: +19.0%
Delta f1 : +19.0%


In [ ]:
# ── Save ──────────────────────────────────────────────────────────────────────
global_model.save('models/bilstm_nsl_kdd_federated.keras')
print('Saved: models/bilstm_nsl_kdd_federated.keras')

Saved: models/bilstm_nsl_kdd_federated.keras
